# METEOR-Net: Google Colab T4 GPU Training Workflow

This notebook provides the verified, reproducible training workflow for **METEOR-Net** on a Google Colab **NVIDIA T4 GPU**.

### Architectural & Operational Safeguards:
1. **Isolated Execution Partition:** Colab writes exclusively to `outputs/colab_t4_smoke_test/` and `outputs/colab_t4_phase8/`. The local Windows CPU baseline in `outputs/phase7b/` is **strictly untouched and frozen**.
2. **Strict Meteorological Event Quarantine:** Training uses Event 1 (Jan 1) and Event 2 (May 20); Validation uses Event 3 (July 19); Event 4 (Sept 7) is **strictly quarantined as held-out test**.
3. **Invariance of Frozen Normalization:** Channel normalization statistics are loaded exclusively from `outputs/normalization/frozen_global_train_stats.json` (SHA-256 `71ef64...`) and are never refit.
4. **Full FP32 Numerical Precision:** Tensors remain in single-precision floating-point (FP32) throughout.
5. **Periodic Live Google Drive Checkpoint Mirroring:** `latest_checkpoint.pt`, `best_model.pt`, and `training_history.json` are written atomically using temporary files and mirrored to Drive at every epoch boundary with post-sync size and existence verification.
6. **Dual-Key Training Execution Guard:** Full training cannot be launched accidentally by "Run All"—it requires both manual confirmation (`CONFIRM_LAUNCH_TRAINING = True`) and verified completion of the GPU smoke test in the current session.

## Stage 1: Explicit Google Drive Authorization & Mount
Mounts Google Drive to access the uploaded dataset archive and maintain persistent checkpoint mirrors.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
assert os.path.exists('/content/drive/MyDrive'), "Google Drive mount failed! Please authorize drive access."
print("[PASS] Google Drive mounted successfully at /content/drive/MyDrive")


## Stage 2: Reproducible Source Ingestion
Ingests and cryptographically verifies the METEOR-Net source code via one of two distinct routes:
* **Route A (Offline Source Bundle on Drive - Recommended):** Verified against its exact cryptographic SHA-256 checksum (`b5f5a051...`). Operates fully offline without Git, network clones, or GitHub credentials.
* **Route B (Private GitHub Clone):** Verified against the pinned, audited Git commit (`c273fc06...`). Uses secure temporary `GIT_ASKPASS` authentication without leaking tokens into CLI arguments, URLs, or command history.

In [ ]:
import os, sys, hashlib, getpass, subprocess, tempfile, stat, shutil

%cd /content
workspace_dir = "/content/meteor_net"

# Route A Verification: Expected Cryptographic SHA-256 for Offline Source Bundle
EXPECTED_BUNDLE_SHA256 = "b5f5a0510a116a24341825a58681146109d8bc27631491418f65723d43deb303"

# Route B Verification: Pinned, Audited Git Commit for GitHub Clone Route
PINNED_GIT_COMMIT = "c273fc0603030d884e5fbb181733ab77080d5339"

drive_source_bundle = "/content/drive/MyDrive/meteor_net_data/meteor_net_source_bundle.zip"

if os.path.exists(drive_source_bundle):
    print(f"[Route A] Found offline source bundle on Drive: {drive_source_bundle}")
    print(f"Verifying source bundle SHA-256 ({os.path.getsize(drive_source_bundle)/(1024*1024):.2f} MB)...")
    h = hashlib.sha256()
    with open(drive_source_bundle, "rb") as f:
        while chunk := f.read(4 * 1024 * 1024):
            h.update(chunk)
    actual_bundle_sha256 = h.hexdigest()
    print(f"Source Bundle SHA-256: {actual_bundle_sha256}")
        # Check authoritative sidecar checksum file if present on Drive alongside the bundle
    drive_sidecar = drive_source_bundle + ".sha256"
    if os.path.exists(drive_sidecar):
        with open(drive_sidecar, "r", encoding="utf-8") as sf:
            sidecar_hash = sf.read().strip().split()[0]
        print(f"Authoritative Sidecar Checksum found: {sidecar_hash}")
        assert actual_bundle_sha256 == sidecar_hash, (
            f"CRITICAL: Bundle SHA-256 does not match authoritative sidecar checksum file!\n"
            f"Got:      {actual_bundle_sha256}\n"
            f"Expected: {sidecar_hash}"
        )
    assert actual_bundle_sha256 == EXPECTED_BUNDLE_SHA256, (
        f"CRITICAL: Source bundle SHA-256 mismatch!\n"
        f"Got:      {actual_bundle_sha256}\n"
        f"Expected: {EXPECTED_BUNDLE_SHA256}"
    )
    if os.path.exists(workspace_dir):
        shutil.rmtree(workspace_dir)
    os.makedirs(workspace_dir, exist_ok=True)
    !unzip -q {drive_source_bundle} -d /content/meteor_net
    print("[PASS] Verified offline source bundle extracted to /content/meteor_net")
else:
    print("[Route B] Source bundle not found on Drive. Proceeding with secure GitHub repository clone...")
    gh_token = getpass.getpass("Enter GitHub Personal Access Token (PAT) for private repo access: ").strip()
    assert gh_token, "GitHub token cannot be empty."

    # Safer temporary authentication via GIT_ASKPASS script with restricted file permissions
    with tempfile.NamedTemporaryFile("w", delete=False) as tf:
        tf.write(gh_token)
        pat_file = tf.name
    os.chmod(pat_file, stat.S_IRUSR | stat.S_IWUSR)

    with tempfile.NamedTemporaryFile("w", delete=False) as af:
        af.write(f'#!/bin/sh\ncat "{pat_file}"\n')
        askpass_file = af.name
    os.chmod(askpass_file, stat.S_IRWXU)

    try:
        env = os.environ.copy()
        env["GIT_ASKPASS"] = askpass_file
        env["GIT_TERMINAL_PROMPT"] = "0"
        if os.path.exists(workspace_dir):
            shutil.rmtree(workspace_dir)
        print("Cloning private repository without leaking token into command line or URL...")
        # Token is passed securely via GIT_ASKPASS; NOT in URL or CLI arguments
        subprocess.run(
            ["git", "clone", "https://x-access-token@github.com/KrishnaManoharReddy907/meteor-net-insat3ds.git", workspace_dir],
            env=env,
            check=True
        )
        # Sanitize remote URL to remove username prefix
        subprocess.run(
            ["git", "remote", "set-url", "origin", "https://github.com/KrishnaManoharReddy907/meteor-net-insat3ds.git"],
            cwd=workspace_dir,
            check=True
        )
    finally:
        if os.path.exists(pat_file):
            os.remove(pat_file)
        if os.path.exists(askpass_file):
            os.remove(askpass_file)
        del gh_token

    current_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=workspace_dir, text=True).strip()
    print(f"Cloned Git Commit: {current_commit}")
    assert current_commit == PINNED_GIT_COMMIT, (
        f"CRITICAL: Cloned commit does not match pinned audited migration commit!\n"
        f"Got:      {current_commit}\n"
        f"Expected: {PINNED_GIT_COMMIT}"
    )
    print(f"[PASS] Pinned Git commit strictly verified: {current_commit}")

%cd /content/meteor_net

# Verify required files are present in workspace root
for req_path in [
    "src/models/meteor_net.py",
    "src/data/goes_dataset.py",
    "scripts/train_meteor_net.py",
    "scripts/run_gpu_smoke_test.py",
    "outputs/dataset_splits/splits_manifest.json",
    "outputs/normalization/frozen_global_train_stats.json",
    "outputs/full_acquisition/data_archive_manifest.json"
]:
    assert os.path.exists(req_path), f"CRITICAL: Workspace root missing essential file: {req_path}"

print("[PASS] Repository root integrity verified.")


## Stage 3: Dataset Ingestion & Deep Integrity Verification (100% Coverage)
1. Verifies the archive SHA-256 before extraction (fail-closed if corrupted).
2. Extracts to local fast NVMe disk (`/content/meteor_net/data/`).
3. Cryptographically audits **all 1,152 extracted NetCDF files** (paths, exact byte sizes, SHA-256 checksums) and confirms no unexpected extra files exist.

In [ ]:
import os, sys, hashlib, json, time, glob

archive_path = "/content/drive/MyDrive/meteor_net_data/data_goes19_4events.tar.gz"
EXPECTED_ARCHIVE_SHA256 = "d640cfdbaf09e9add5e8bfb0d82ddb63a97c30f4c09807e4ded222ccf0975aa5"
EXPECTED_ARCHIVE_BYTES = 4192771771
EXPECTED_FILE_COUNT = 1152

assert os.path.exists(archive_path), (
    f"Dataset archive not found at: {archive_path}\n"
    "Please upload 'data_goes19_4events.tar.gz' to 'My Drive/meteor_net_data/'."
)

archive_bytes = os.path.getsize(archive_path)
assert archive_bytes == EXPECTED_ARCHIVE_BYTES, (
    f"Archive size mismatch!\nGot: {archive_bytes} bytes\nExpected: {EXPECTED_ARCHIVE_BYTES} bytes"
)

print(f"Verifying dataset archive SHA-256 ({archive_bytes / (1024**3):.3f} GB)...")
t0 = time.perf_counter()
h = hashlib.sha256()
with open(archive_path, "rb") as f:
    while chunk := f.read(4 * 1024 * 1024):
        h.update(chunk)
actual_archive_sha256 = h.hexdigest()
t_hash_archive = time.perf_counter() - t0
print(f"Archive SHA-256: {actual_archive_sha256} (computed in {t_hash_archive:.1f} s)")
assert actual_archive_sha256 == EXPECTED_ARCHIVE_SHA256, (
    f"Archive corruption detected!\nGot:      {actual_archive_sha256}\nExpected: {EXPECTED_ARCHIVE_SHA256}"
)
print("[PASS] Dataset archive cryptographic checksum verified.")

print("\nExtracting dataset to local high-speed VM storage...")
t_extract_0 = time.perf_counter()
!tar -xzf {archive_path} -C /content/meteor_net/
print(f"Extraction completed in {time.perf_counter() - t_extract_0:.1f} s.")

# 100% Deep Per-File Integrity Audit
manifest_file = "outputs/full_acquisition/data_archive_manifest.json"
assert os.path.exists(manifest_file), f"Archive manifest missing: {manifest_file}"
with open(manifest_file, "r") as f:
    archive_manifest = json.load(f)

expected_files = archive_manifest["files"]
assert len(expected_files) == EXPECTED_FILE_COUNT, f"Manifest must contain {EXPECTED_FILE_COUNT} files, found {len(expected_files)}"

# Scan all NetCDF files on disk to ensure no unexpected extra files exist
found_nc_files = []
for root, _, files in os.walk("/content/meteor_net/data"):
    for f in files:
        if f.endswith(".nc"):
            rel_p = os.path.relpath(os.path.join(root, f), "/content/meteor_net").replace("\\", "/")
            found_nc_files.append(rel_p)

found_nc_set = set(found_nc_files)
expected_set = set(expected_files.keys())

missing_files = sorted(list(expected_set - found_nc_set))
unexpected_files = sorted(list(found_nc_set - expected_set))

assert len(missing_files) == 0, f"Missing {len(missing_files)} files: {missing_files[:5]}"
assert len(unexpected_files) == 0, f"Unexpected extra NetCDF files detected: {unexpected_files[:5]}"
assert len(found_nc_files) == EXPECTED_FILE_COUNT, f"Expected {EXPECTED_FILE_COUNT} files, found {len(found_nc_files)}"

print(f"\nAuditing all {len(expected_files)} extracted NetCDF files (100% cryptographic SHA-256 check)...")
t_audit_0 = time.perf_counter()
corrupt_files = []

for idx, (rel_path, meta) in enumerate(expected_files.items()):
    local_path = os.path.join("/content/meteor_net", rel_path)
    # 1. Byte size verification
    if os.path.getsize(local_path) != meta["size_bytes"]:
        corrupt_files.append((rel_path, "size_mismatch"))
        continue
    # 2. SHA-256 verification (all 1,152 files)
    fh = hashlib.sha256()
    with open(local_path, "rb") as f:
        while chunk := f.read(1024 * 1024):
            fh.update(chunk)
    if fh.hexdigest() != meta["sha256"]:
        corrupt_files.append((rel_path, "sha256_mismatch"))
    if (idx + 1) % 200 == 0 or (idx + 1) == len(expected_files):
        print(f"  Verified {idx + 1:04d}/{len(expected_files)} files...")

t_audit = time.perf_counter() - t_audit_0
assert len(corrupt_files) == 0, f"Corrupt files detected: {corrupt_files[:5]}"
print(f"[PASS] All {len(expected_files)} NetCDF files verified intact with exact sizes and SHA-256 hashes in {t_audit:.1f} s.")


## Stage 4: Dependencies Installation & CUDA Hardware Inspection
Installs missing third-party utilities (`requirements.txt`) without modifying Colab's pre-installed CUDA PyTorch build.

In [ ]:
!pip install -q -r requirements.txt

import torch, h5py, matplotlib, psutil
assert torch.cuda.is_available(), (
    "CRITICAL: CUDA is not available!\n"
    "In Colab, go to Runtime -> Change runtime type -> Hardware accelerator: T4 GPU -> Save."
)
device_name = torch.cuda.get_device_name(0)
free_vram, total_vram = torch.cuda.mem_get_info(0)

print(f"PyTorch Version:     {torch.__version__} (CUDA: {torch.version.cuda})")
print(f"h5py Version:        {h5py.__version__}")
print(f"matplotlib Version:  {matplotlib.__version__}")
print(f"psutil Version:      {psutil.__version__}")
print(f"GPU Hardware:        {device_name}")
print(f"CUDA Compute Cap:    {torch.cuda.get_device_capability(0)}")
print(f"GPU VRAM Available:  {free_vram / (1024**3):.2f} GB free / {total_vram / (1024**3):.2f} GB total")
!nvidia-smi


## Stage 5: Cryptographic Integrity & Normalization Provenance Check
Verifies SHA-256 hashes for dataset split manifest, frozen normalization statistics, and acquisition inventory.

In [ ]:
import hashlib, json

def verify_sha256(filepath, expected_hash):
    with open(filepath, "rb") as f:
        actual_hash = hashlib.sha256(f.read()).hexdigest()
    assert actual_hash == expected_hash, f"SHA-256 mismatch for {filepath}:\n  Got:      {actual_hash}\n  Expected: {expected_hash}"
    print(f"[VERIFIED] {filepath}: {actual_hash}")

print("Verifying critical manifest hashes...")
verify_sha256("outputs/dataset_splits/splits_manifest.json", "6a173db4760c49639f4b830f6814f97d2d970a1da48c9d62e19c81d20bbd7d7f")
verify_sha256("outputs/normalization/frozen_global_train_stats.json", "71ef64929a8875490a24cbd33e8aec87e14349cb966d32d0f68a28abc51c4164")
verify_sha256("outputs/full_acquisition/acquisition_inventory.json", "26cf302690ce3877d6e8923e5ea4141304447a5e81065adfc244e9c8ff6b4de7")

# Verify test event quarantine
with open("outputs/dataset_splits/splits_manifest.json", "r") as f:
    manifest = json.load(f)
assert manifest["evaluation_validity"]["group_to_split_mapping"]["event_2026-09-07"] == "test"
print("[VERIFIED] Event 4 (September 7, 2026) is strictly quarantined as 'test'.")


## Stage 6: Standalone GPU Smoke Test (Manually Initiated Preflight)
Executes `scripts/run_gpu_smoke_test.py` to test the real model, frozen normalization, convective loss, forward pass, autograd backward pass, AdamW step, and atomic checkpoint serialization on CUDA.
Upon successful execution, records session authorization flags required by Stage 7.

In [ ]:
import os, json, subprocess

smoke_output_dir = "outputs/colab_t4_smoke_test"
smoke_results_path = os.path.join(smoke_output_dir, "gpu_smoke_test_results.json")

# Clean stale smoke test outputs prior to run
if os.path.exists(smoke_results_path):
    os.remove(smoke_results_path)

print("Executing Standalone GPU Smoke Test on T4 GPU...")
subprocess.run(
    ["python", "scripts/run_gpu_smoke_test.py", "--device", "cuda", "--output_dir", smoke_output_dir],
    check=True
)

assert os.path.isfile(smoke_results_path), f"Smoke test output missing: {smoke_results_path}"
with open(smoke_results_path, "r") as f:
    smoke_summary = json.load(f)

print("\n--- GPU Smoke Test Results ---")
print(json.dumps(smoke_summary, indent=2))

assert smoke_summary["status"] == "PASS", "GPU Smoke Test FAILED! Do not proceed to full training."
assert smoke_summary["hardware"]["cuda_available"] is True, "Smoke test did not run on CUDA!"
assert smoke_summary["hardware"]["device"].startswith("cuda"), "Smoke test device is not CUDA!"

# Authorize in-kernel session flags
GLOBAL_SMOKE_TEST_PASSED = True
GLOBAL_SMOKE_TEST_TIMESTAMP = smoke_summary["timestamp"]
GLOBAL_SMOKE_TEST_DEVICE = smoke_summary["hardware"]["device"]

print(f"\n[PASS] Verified GPU Smoke Test on {GLOBAL_SMOKE_TEST_DEVICE} at {GLOBAL_SMOKE_TEST_TIMESTAMP}.")


## Stage 7: Full Controlled Training Runner (Natural-Ratio Single-Pass Full Training)
Target output directory: `outputs/colab_t4_full_training/`.
Persistent Google Drive mirror: `/content/drive/MyDrive/meteor_net_runs/colab_t4_full_training/`.

> **CONFIGURATION & SAMPLER DETAILS:**
> - **Sampler Mode:** `natural_single_pass` (visits all 6,480 training patches exactly once per epoch, preserving ~35:65 natural convective/ambient distribution)
> - **Validation Mode:** `full_event` (evaluates all 3,240 validation patches from July 19, 2026 event at every epoch boundary)
> - **Gradient Accumulation:** 16 microbatches per optimizer update -> 405 updates per epoch, 2,025 total scheduled updates over 5 epochs
> - **Gradient Normalization:** Unbiased dynamic group scaling (`loss / current_group_size`)
> - **Warmup:** 162 updates (8% of total scheduled updates) + Cosine Decay to 1e-6
> - **Quarantine:** September 7, 2026 test event remains completely excluded from training, validation, model selection, and normalization

> **SAFETY GUARDS:**
> 1. Set `CONFIRM_LAUNCH_TRAINING = True` below only when you intend to launch training.
> 2. Requires verified completion of Stage 6 (GPU Smoke Test) in this active session.


In [ ]:
# ==============================================================================
# SAFETY GUARD: DUAL-KEY AUTHORIZATION REQUIRED BEFORE FULL TRAINING
# ==============================================================================
# Key 1: Explicit user confirmation boolean (prevents accidental Run All)
CONFIRM_LAUNCH_TRAINING = False  # <-- Change to True ONLY to launch training

if not CONFIRM_LAUNCH_TRAINING:
    raise RuntimeError(
        "TRAINING GUARD BLOCKED: 'CONFIRM_LAUNCH_TRAINING' is False.\n"
        "This cell will not execute unless explicitly confirmed.\n"
        "Verify that Stage 6 (GPU Smoke Test) passed on your T4 GPU, "
        "then change CONFIRM_LAUNCH_TRAINING = True to execute."
    )

# Key 2: In-Kernel Session Guard (requires Stage 6 executed in current session)
if 'GLOBAL_SMOKE_TEST_PASSED' not in globals() or not GLOBAL_SMOKE_TEST_PASSED:
    raise RuntimeError(
        "TRAINING GUARD BLOCKED: GPU Smoke Test (Stage 6) has NOT been executed or did not pass in this session!\n"
        "You must execute Stage 6 in the current session before launching full training."
    )

smoke_results_path = "outputs/colab_t4_smoke_test/gpu_smoke_test_results.json"
assert os.path.isfile(smoke_results_path), "Smoke test results file missing!"
with open(smoke_results_path, "r") as f:
    _check_smoke = json.load(f)
assert _check_smoke.get("status") == "PASS", "Smoke test recorded failure!"
assert _check_smoke.get("timestamp") == GLOBAL_SMOKE_TEST_TIMESTAMP, "Stale smoke test output detected!"
assert _check_smoke.get("hardware", {}).get("cuda_available") is True, "Smoke test was not executed on CUDA!"

print("[GUARD VERIFIED] Smoke test passed on GPU in this session. Launching authorized full-data training (natural_single_pass)...")

!python scripts/train_meteor_net.py \
    --device cuda \
    --output_dir outputs/colab_t4_full_training \
    --drive_sync_dir /content/drive/MyDrive/meteor_net_runs/colab_t4_full_training \
    --epochs 5 \
    --sampler_mode natural_single_pass \
    --val_mode full_event \
    --grad_accum_steps 16 \
    --lr 1e-4 \
    --weight_decay 1e-4 \
    --seed 42 \
    --early_stopping_patience 2


## Stage 8: Checkpoint Recovery & Live Resume Procedure
If your Colab session disconnected or timed out during full training, execute this cell to resume training from the latest full-training Drive checkpoint.

> **CRITICAL RESTRICTION — PILOT CHECKPOINT CANNOT RESUME FULL-DATA TRAINING:**
> The Phase 8 pilot checkpoint (`outputs/colab_t4_phase8/latest_checkpoint.pt`) must **NEVER** be used to resume full-data training. Resumption across different sampler modes (`balanced_pilot` vs `natural_single_pass`), mismatched microbatch budgets, or mismatched gradient accumulation steps is strictly rejected fail-closed. Full-data training must be resumed exclusively using `latest_checkpoint.pt` from `/content/drive/MyDrive/meteor_net_runs/colab_t4_full_training/`.

### Resume Safeguards & Preflight:
1. **Hardware Verification:** Asserts that CUDA is available before launching the resume runner.
2. **Checkpoint Deserialization & Preflight:** Verifies that the Drive checkpoint exists and is fully readable, reporting the saved epoch, optimizer update count, microbatch count, best validation loss, and sampler mode.
3. **Sampler Consistency & Incompatibility Check:** Strictly verifies that the checkpoint sampler mode matches `--sampler_mode natural_single_pass`, rejecting incompatible cross-sampler resume attempts.
4. **Artifact Immutability:** Existing untrained baseline metrics (`baseline_metrics.json`) and preflight serialization tests (`preflight_checkpoint_test.pt`) are **preserved without overwriting**.
5. **Full State Restoration:** Restores model weights, optimizer state, scheduler state, RNG states, and training history, resuming at microbatch 0 of the interrupted epoch.


In [ ]:
# ==============================================================================
# STAGE 8: CHECKPOINT RESUME CELL (RUN ONLY IF PREVIOUS SESSION DISCONNECTED)
# ==============================================================================
CONFIRM_RESUME_TRAINING = False  # <-- Change to True ONLY after reviewing preflight below

import os, torch

# 1. Hardware Preflight: Ensure CUDA is available
assert torch.cuda.is_available(), (
    "CRITICAL: CUDA is not available for resume!\n"
    "In Colab, go to Runtime -> Change runtime type -> Hardware accelerator: T4 GPU -> Save."
)
print(f"[Resume Preflight] GPU Hardware: {torch.cuda.get_device_name(0)} (CUDA compute {torch.cuda.get_device_capability(0)})")

# 2. Checkpoint Preflight: Ensure Drive checkpoint exists and is readable
resume_ckpt = "/content/drive/MyDrive/meteor_net_runs/colab_t4_full_training/latest_checkpoint.pt"
assert os.path.isfile(resume_ckpt), f"Resume checkpoint not found: {resume_ckpt}"

print(f"[Resume Preflight] Inspecting Drive checkpoint: {resume_ckpt} ({os.path.getsize(resume_ckpt)/(1024*1024):.2f} MB)...")
try:
    ckpt_preview = torch.load(resume_ckpt, map_location="cpu")
    res_epoch = ckpt_preview.get("epoch")
    res_updates = ckpt_preview.get("optimizer_update_count")
    res_microbatches = ckpt_preview.get("microbatch_count")
    res_best_loss = ckpt_preview.get("best_val_loss")
    print(f"  [PASS] Checkpoint deserialized successfully.")
    print(f"  [State Preview] Saved Epoch: {res_epoch} | Updates: {res_updates} | MBs: {res_microbatches} | Best Val Loss: {res_best_loss} K")
    print(f"  [Target] Training will safely resume at microbatch 0 of Epoch {res_epoch + 1}.")
    del ckpt_preview
except Exception as e:
    raise RuntimeError(f"CRITICAL: Checkpoint {resume_ckpt} is unreadable or corrupted: {e}")

if not CONFIRM_RESUME_TRAINING:
    raise RuntimeError(
        "RESUME GUARD BLOCKED: 'CONFIRM_RESUME_TRAINING' is False.\n"
        "Review the preflight output above, then set CONFIRM_RESUME_TRAINING = True to resume training."
    )

print(f"\nLaunching authorized full-data resume training from: {resume_ckpt}")
!python scripts/train_meteor_net.py \
    --device cuda \
    --output_dir outputs/colab_t4_full_training \
    --drive_sync_dir /content/drive/MyDrive/meteor_net_runs/colab_t4_full_training \
    --epochs 5 \
    --sampler_mode natural_single_pass \
    --val_mode full_event \
    --grad_accum_steps 16 \
    --lr 1e-4 \
    --weight_decay 1e-4 \
    --seed 42 \
    --early_stopping_patience 2 \
    --resume {resume_ckpt}


## Stage 9: Final Artifacts Audit in Google Drive
Audits the persistent experiment outputs copied to Google Drive upon run completion.

In [ ]:
import os, json

drive_dest = "/content/drive/MyDrive/meteor_net_runs/colab_t4_full_training"
if not os.path.exists(drive_dest):
    print(f"[Note] Full-training directory {drive_dest} not yet created; inspecting completed pilot: colab_t4_phase8")
    drive_dest = "/content/drive/MyDrive/meteor_net_runs/colab_t4_phase8"

print(f"Auditing live-synced artifacts in Google Drive: {drive_dest}")
assert os.path.exists(drive_dest), f"Drive destination does not exist: {drive_dest}"

synced_files = os.listdir(drive_dest)
for f in synced_files:
    f_path = os.path.join(drive_dest, f)
    print(f"  [Drive File] {f} ({os.path.getsize(f_path)/(1024*1024):.2f} MB)")

assert "latest_checkpoint.pt" in synced_files, "Missing latest_checkpoint.pt in Drive!"
assert "training_history.json" in synced_files, "Missing training_history.json in Drive!"

with open(os.path.join(drive_dest, "training_history.json"), "r") as f:
    history = json.load(f)

print(f"\nTraining History Summary:")
print(f"  Total Epochs Completed: {len(history.get('epochs', []))}")
print(f"  Best Validation Loss:   {history.get('best_val_loss')} K")
for ep in history.get("epochs", []):
    print(f"    Epoch {ep['epoch']}: Train {ep['train_loss_kelvin']} K | Val {ep['val_loss_kelvin']} K | LR {ep['lr']:.2e} | Duration {ep['duration_s']}s")

print(f"\n[PASS] All experiment artifacts in {drive_dest} successfully verified in Google Drive.")


## Stage 10: Held-Out Test Evaluation (September 7, 2026 Event)
Performs rigorous, scientific benchmark evaluation of trained METEOR-Net against the quarantined, held-out test event (`event_2026-09-07`).

### Evaluation Protocols & Scientific Guardrails:
1. **Strict Split Isolation:** Evaluates strictly on `event_2026-09-07` (54 temporal sequence windows, 3,240 patches of size 128x128 with stride 64), which was untouched during training and validation.
2. **Fair Physical Baselines:** Compares METEOR-Net against:
   - **Linear Temporal Interpolation:** $0.5 \times X[t=0] + 0.5 \times X[t=+30]$
   - **Persistence ($t=0$ min):** $X[t=0]$
   - **Persistence ($t=+30$ min):** $X[t=+30]$
   All methods evaluate on the exact same target timestamps (+15 min), valid pixels, and physical Kelvin scale.
3. **Overlap-Aware Scene Metrics:** In addition to patch-level metrics, reconstructs the full cropped observation grid ($453 \times 740$) across overlapping patches to eliminate duplicate pixel counting.
4. **Valid Pixel & Mask Integrity:** Evaluates only pixels with `combined_mask == True` across all 4 bands and timeframes; non-finite/invalid values are strictly excluded.
5. **SSIM & PSNR:** Evaluated exclusively on fully valid patches ($128 \times 128$) using training-derived physical dynamic ranges ($6\sigma$).
6. **Dual-Key Safety Guard:** Requires manual confirmation (`CONFIRM_EVALUATE_TEST = True`) before execution.

In [ ]:
# ==============================================================================
# STAGE 10: HELD-OUT TEST EVALUATION CELL (MANUAL EXECUTION ONLY)
# ==============================================================================
CONFIRM_EVALUATE_TEST = False  # <-- Change to True ONLY after reviewing preflight below

import os, json, torch

# Evaluation Target Mode: 'pilot' (Phase 8 completed run) or 'full_training' (Full-dataset run)
# Set EVAL_TARGET = "pilot" to evaluate the completed 5-epoch pilot checkpoint.
# Set EVAL_TARGET = "full_training" after full-data training completes.
EVAL_TARGET = "pilot"

if EVAL_TARGET == "pilot":
    ckpt_path = "/content/drive/MyDrive/meteor_net_runs/colab_t4_phase8/best_model.pt"
    local_out_dir = "outputs/colab_t4_phase8_test_evaluation"
    drive_sync_dir = "/content/drive/MyDrive/meteor_net_runs/colab_t4_phase8_test_evaluation"
elif EVAL_TARGET == "full_training":
    ckpt_path = "/content/drive/MyDrive/meteor_net_runs/colab_t4_full_training/best_model.pt"
    local_out_dir = "outputs/colab_t4_full_training_test_evaluation"
    drive_sync_dir = "/content/drive/MyDrive/meteor_net_runs/colab_t4_full_training_test_evaluation"
else:
    raise ValueError(f"Unknown EVAL_TARGET: {EVAL_TARGET}. Must be 'pilot' or 'full_training'.")

norm_path = "outputs/normalization/frozen_global_train_stats.json"
split_path = "outputs/dataset_splits/splits_manifest.json"

assert os.path.isfile(ckpt_path), f"CRITICAL: Expected checkpoint not found at {ckpt_path}"
assert os.path.isfile(norm_path), f"CRITICAL: Frozen normalization stats not found at {norm_path}"
assert os.path.isfile(split_path), f"CRITICAL: Splits manifest not found at {split_path}"

# Assert split integrity
with open(split_path, "r") as f:
    splits_data = json.load(f)
assert "test" in splits_data, "Splits manifest missing 'test' partition!"
assert "event_2026-09-07" in splits_data["test"].get("events", []), (
    "CRITICAL: Test split must contain event_2026-09-07!"
)

# Assert normalization integrity
with open(norm_path, "r") as f:
    norm_data = json.load(f)
assert "means" in norm_data and "stds" in norm_data, "Invalid normalization stats!"

# Assert safe output directory (must not overwrite training or baseline directories)
protected = [
    "outputs/phase7b",
    "outputs/normalization",
    "outputs/dataset_splits",
    "outputs/colab_t4_smoke_test",
    "outputs/colab_t4_phase8",
    "outputs/colab_t4_full_training"
]
for p in protected:
    assert not os.path.abspath(local_out_dir).startswith(os.path.abspath(p)), f"Output dir inside protected {p}!"

print("[Stage 10 Preflight] All artifacts verified:")
print(f"  Target Mode:   {EVAL_TARGET.upper()} EVALUATION")
print(f"  Checkpoint:    {ckpt_path} ({os.path.getsize(ckpt_path)/(1024*1024):.2f} MB)")
print(f"  Normalization: {norm_path}")
print(f"  Test Event:    event_2026-09-07 (quarantined held-out)")
print(f"  Local Out Dir: {local_out_dir}")
print(f"  Drive Sync:    {drive_sync_dir}")
print(f"  CUDA Device:   {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None (CPU)'}")

if not CONFIRM_EVALUATE_TEST:
    raise RuntimeError(
        "TEST EVALUATION GUARD BLOCKED: 'CONFIRM_EVALUATE_TEST' is False.\n"
        "Review the preflight output above, then set CONFIRM_EVALUATE_TEST = True to execute evaluation."
    )

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nLaunching Held-Out Test Evaluation ({EVAL_TARGET}) on device: {device}...")
!python scripts/evaluate_meteor_net_test.py \
    --checkpoint {ckpt_path} \
    --manifest {split_path} \
    --stats {norm_path} \
    --output_dir {local_out_dir} \
    --drive_sync_dir {drive_sync_dir} \
    --device {device}

# Assert evaluation outputs exist
summary_file = os.path.join(local_out_dir, "test_evaluation_summary.md")
metrics_file = os.path.join(local_out_dir, "test_evaluation_metrics.json")
assert os.path.isfile(metrics_file), f"Evaluation failed to create {metrics_file}"
assert os.path.isfile(summary_file), f"Evaluation failed to create {summary_file}"

print("\n" + "="*80)
print("STAGE 10 EVALUATION COMPLETE - DISPLAYING SUMMARY:")
print("="*80 + "\n")
with open(summary_file, "r") as f:
    print(f.read())

print(f"\n[PASS] Evaluation artifacts successfully written to:")
print(f"  Local:  {local_out_dir}")
print(f"  Drive:  {drive_sync_dir}")
